# KisanSetu — Crop Price Prediction (Full Pipeline)

Cell-by-cell ML pipeline: Google Drive se data load karna, cleaning, EDA, aur GPU-supported XGBoost model se Min/Max/Modal price predict karna.

**Dataset size:** ~2,78,49,344 rows — isliye memory-safe loading, chart ke liye sampling, aur practical cross-validation use kiya gaya hai.

## Cell 1 — Google Drive link se CSV download
Isko run karne ke baad file `/content` folder me aa jaani chahiye.

In [ ]:
!pip install -q gdown

import gdown

# Apna Google Drive file ID yahan daalo
file_id = "YOUR_FILE_ID_HERE"
output_path = "/content/kisan_setu_data.csv"

gdown.download(f"https://drive.google.com/uc?id={file_id}", output_path, quiet=False)


## Cell 2 — Libraries import

In [ ]:
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb
import joblib

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 50)


## Cell 3 — Dataset load karke `df` banana
Dataset me 27,849,344 rows hain, isliye ye cell RAM kaafi use karega. Column dtypes explicitly set kiye hain taaki memory usage kam rahe.

_Sample 10 rows print nahi kar rahe, jaisa requirement tha._

In [ ]:
csv_path = "/content/kisan_setu_data.csv"

# Memory-safe dtypes — categorical columns "category" me, prices float32 me
dtype_map = {
    "State": "category",
    "District": "category",
    "Market": "category",
    "Commodity": "category",
    "Variety": "category",
    "Min_Price": "float32",
    "Max_Price": "float32",
    "Modal_Price": "float32",
}

df = pd.read_csv(csv_path, dtype=dtype_map, low_memory=False)

print(f"Rows: {len(df):,}  Columns: {df.shape[1]}")


## Cell 4 — Dataset info

In [ ]:
df.info(memory_usage="deep")


## Cell 5 — Descriptive statistics

In [ ]:
df.describe(include="all").T


## Cell 6 — Column names aur shape

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(list(df.columns))


---
## Data Cleaning

## Cell 7 — `Arrival_Date` convert + 4 columns add
Year, Month, Day, DayOfWeek banayenge. Original `Arrival_Date` delete ho jayega.

In [ ]:
df["Arrival_Date"] = pd.to_datetime(df["Arrival_Date"], dayfirst=True, errors="coerce")

df["Year"] = df["Arrival_Date"].dt.year.astype("int16")
df["Month"] = df["Arrival_Date"].dt.month.astype("int8")
df["Day"] = df["Arrival_Date"].dt.day.astype("int8")
df["DayOfWeek"] = df["Arrival_Date"].dt.dayofweek.astype("int8")

# Original date column delete
df.drop(columns=["Arrival_Date"], inplace=True)

df[["Year", "Month", "Day", "DayOfWeek"]].head()


## Cell 8 — Price columns clean karna
Logic:
- Positive value valid hai.
- Row me invalid price hai to usi row ke baaki positive prices ka median use hoga.
- Row me koi bhi positive price nahi hai to poore dataset ke valid positive prices ka global median use hoga.
- NaN ya non-numeric value bhi isi tarah clean hogi.

In [ ]:
price_cols = ["Min_Price", "Max_Price", "Modal_Price"]

# Step 1: non-numeric / negative / zero -> NaN
for col in price_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df.loc[df[col] <= 0, col] = np.nan

# Step 2: global medians (sirf positive valid values se) — fallback ke liye
global_medians = {col: df.loc[df[col] > 0, col].median() for col in price_cols}
print("Global medians:", global_medians)

# Step 3 (vectorized, 2.78 crore rows ke liye row-wise apply bahut slow hoga):
# pehle row-wise median se fill, phir jo bacha usse global median se fill
row_median = df[price_cols].median(axis=1, skipna=True)

for col in price_cols:
    mask_nan = df[col].isna()
    df.loc[mask_nan, col] = row_median[mask_nan]
    still_nan = df[col].isna()
    df.loc[still_nan, col] = global_medians[col]

del row_median
gc.collect()

df[price_cols].describe()


## Cell 9 — Check karo cleaning sahi hui ya nahi

In [ ]:
print("Remaining nulls in price columns:")
print(df[price_cols].isna().sum())

print("\nMin values after cleaning:")
print(df[price_cols].min())

print("\nAny zero/negative left?")
print((df[price_cols] <= 0).sum())


---
## Unique Values aur Null-like Values

## Cell 10 — State unique values

In [ ]:
print("Unique States:", df["State"].nunique())
print(df["State"].unique())


## Cell 11 — Commodity aur District unique count

In [ ]:
print("Unique Commodities:", df["Commodity"].nunique())
print("Unique Districts:", df["District"].nunique())

print("\nTop 10 commodities by count:")
print(df["Commodity"].value_counts().head(10))

print("\nTop 10 districts by count:")
print(df["District"].value_counts().head(10))


## Cell 12 — Null, NaN, NA, blank jaise values check
`info()` se har tarah ke string null values nahi pata chalte, isliye alag se check kar rahe hain.

In [ ]:
null_like_values = ["", " ", "NA", "N/A", "null", "NULL", "None", "none", "-", "?"]

null_like_report = {}
for col in df.select_dtypes(include=["category", "object"]).columns:
    count = df[col].astype(str).str.strip().isin(null_like_values).sum()
    null_like_report[col] = count

print("Null-like string values per column:")
for col, count in null_like_report.items():
    print(f"{col}: {count}")


## Cell 13 — Actual null values report

In [ ]:
null_report = df.isna().sum()
null_pct = (null_report / len(df)) * 100

null_summary = pd.DataFrame({"null_count": null_report, "null_pct": null_pct})
null_summary[null_summary["null_count"] > 0].sort_values("null_pct", ascending=False)


---
## Charts and Analysis
2.78 crore rows ko directly chart karna bahut heavy hoga, isliye chart ke liye 100,000 rows ka representative sample lenge. Model training ke data ko sample nahi kar rahe.

## Cell 14 — Visualization sample

In [ ]:
SAMPLE_SIZE = 100_000
df_sample = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42)
print("Sample shape:", df_sample.shape)


## Cell 15 — Categorical charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

top_states = df_sample["State"].value_counts().head(10)
sns.barplot(x=top_states.values, y=top_states.index, ax=axes[0], palette="viridis")
axes[0].set_title("Top 10 States by Record Count (sample)")

top_commodities = df_sample["Commodity"].value_counts().head(10)
sns.barplot(x=top_commodities.values, y=top_commodities.index, ax=axes[1], palette="magma")
axes[1].set_title("Top 10 Commodities by Record Count (sample)")

plt.tight_layout()
plt.show()


## Cell 16 — Numerical price charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, price_cols):
    sns.histplot(df_sample[col], bins=50, kde=True, ax=ax)
    ax.set_title(f"Distribution of {col} (sample)")

plt.tight_layout()
plt.show()


## Cell 17 — Correlation heatmap

In [ ]:
numeric_cols = price_cols + ["Year", "Month", "Day", "DayOfWeek"]
corr = df_sample[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap (sample)")
plt.show()


## Cell 18 — Commodity-wise price table

In [ ]:
commodity_price_table = (
    df_sample.groupby("Commodity")[price_cols]
    .mean()
    .sort_values("Modal_Price", ascending=False)
)
commodity_price_table.head(20)


---
## ML Model Preparation

**Input features:** State, District, Market, Commodity, Variety, Year, Month, Day, DayOfWeek

**Output:** Min_Price, Max_Price, Modal_Price

Current day's Min/Max/Modal ko input me nahi le rahe, kyunki unhi values ko predict karna hai — warna data leakage ho jayega.

## Cell 19 — Feature and target setup

In [ ]:
feature_cols = [
    "State", "District", "Market", "Commodity", "Variety",
    "Year", "Month", "Day", "DayOfWeek",
]
target_cols = price_cols  # Min_Price, Max_Price, Modal_Price

X = df[feature_cols].copy()
y = df[target_cols].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)


## Cell 20 — Train/test split
Time-series data hai, isliye random split ke bajaye chronological split use karenge. Original `Arrival_Date` delete ho chuki hai, isliye Year/Month/Day se sort key banate hain.

In [ ]:
sort_key = (
    df["Year"].astype(int) * 10000
    + df["Month"].astype(int) * 100
    + df["Day"].astype(int)
)

order = sort_key.sort_values().index
split_idx = int(len(order) * 0.8)

train_idx = order[:split_idx]
test_idx = order[split_idx:]

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


## Cell 21 — Preprocessing pipeline
Yahi null filling + categorical encoding karegi.

In [ ]:
categorical_features = ["State", "District", "Market", "Commodity", "Variety"]
numeric_features = ["Year", "Month", "Day", "DayOfWeek"]

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

preprocessor = ColumnTransformer(transformers=[
    ("cat", categorical_pipeline, categorical_features),
    ("num", numeric_pipeline, numeric_features),
])


---
## Model

## Cell 22 — GPU XGBoost model
3 outputs (Min/Max/Modal) ke liye `MultiOutputRegressor` ke saath XGBoost use kar rahe hain. GPU available nahi hai to `device` ko `"cpu"` kar dena.

In [ ]:
xgb_model = xgb.XGBRegressor(
    tree_method="hist",
    device="cuda",         # GPU training; agar GPU nahi hai to "cpu" kar do
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

multi_output_model = MultiOutputRegressor(xgb_model)


## Cell 23 — Final ML pipeline

In [ ]:
final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", multi_output_model),
])


## Cell 24 — Model training
**Note:** Agar GPU ya RAM error aata hai, is cell ko baar-baar repeat mat karna — error message bhejna. 2.78 crore rows par 3-output XGBoost training time aur RAM requirement kaafi high ho sakti hai.

In [ ]:
final_pipeline.fit(X_train, y_train)
print("Training complete.")


---
## Cross-Validation

## Cell 25 — K-fold cross-validation
5-fold CV. Full 2.78 crore rows par 5 complete XGBoost training runs bahut heavy honge, isliye validation ke liye bounded sample use kar rahe hain. Final model upar poore train data par trained hai.

In [ ]:
CV_SAMPLE_SIZE = 200_000
cv_sample_idx = X_train.sample(n=min(CV_SAMPLE_SIZE, len(X_train)), random_state=42).index

X_cv = X_train.loc[cv_sample_idx]
y_cv = y_train.loc[cv_sample_idx]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_cv), start=1):
    X_tr, X_val = X_cv.iloc[tr_idx], X_cv.iloc[val_idx]
    y_tr, y_val = y_cv.iloc[tr_idx], y_cv.iloc[val_idx]

    fold_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", MultiOutputRegressor(xgb_model)),
    ])
    fold_pipeline.fit(X_tr, y_tr)
    preds = fold_pipeline.predict(X_val)
    fold_mae = mean_absolute_error(y_val, preds)
    cv_scores.append(fold_mae)
    print(f"Fold {fold}: MAE = {fold_mae:.2f}")

print(f"\nAverage CV MAE: {np.mean(cv_scores):.2f}")


---
## Model Testing

## Cell 26 — 3 output predictions + metrics

In [ ]:
y_pred = final_pipeline.predict(X_test)
y_pred_df = pd.DataFrame(y_pred, columns=target_cols, index=y_test.index)

metrics_report = {}
for col in target_cols:
    mae = mean_absolute_error(y_test[col], y_pred_df[col])
    rmse = np.sqrt(mean_squared_error(y_test[col], y_pred_df[col]))
    r2 = r2_score(y_test[col], y_pred_df[col])
    metrics_report[col] = {"MAE": mae, "RMSE": rmse, "R2": r2}

pd.DataFrame(metrics_report).T


## Cell 27 — Actual vs predicted price chart

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

plot_n = 200  # readability ke liye sirf pehle 200 test points plot karenge
for ax, col in zip(axes, target_cols):
    ax.plot(y_test[col].values[:plot_n], label="Actual", marker="o", markersize=3)
    ax.plot(y_pred_df[col].values[:plot_n], label="Predicted", marker="x", markersize=3)
    ax.set_title(f"Actual vs Predicted — {col}")
    ax.legend()

plt.tight_layout()
plt.show()


## Cell 28 — Actual vs predicted scatter charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, target_cols):
    ax.scatter(y_test[col], y_pred_df[col], alpha=0.3, s=8)
    lims = [
        min(y_test[col].min(), y_pred_df[col].min()),
        max(y_test[col].max(), y_pred_df[col].max()),
    ]
    ax.plot(lims, lims, "r--")
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.set_title(col)

plt.tight_layout()
plt.show()


---
## Save Model

## Cell 29 — Model save

In [ ]:
joblib.dump(final_pipeline, "/content/kisan_setu_price_model.pkl")
print("Model saved to /content/kisan_setu_price_model.pkl")


## Cell 30 — Final 3-column output

In [ ]:
final_output = y_pred_df[["Min_Price", "Max_Price", "Modal_Price"]].copy()
final_output.reset_index(drop=True, inplace=True)
final_output.head(10)


---
## Ek Important Correction

Ye notebook baseline model hai — direct reliable 10-day forecast model nahi, kyunki isme previous days ke price lags shaamil nahi hain.

**KisanSetu ke final version me aage add karna hai:**
- Previous 1, 3, 7, 14, 30 days ke price lags
- Market + Commodity + Variety ke hisaab se time-series grouping
- 10-day recursive forecast
- Min/Max/Modal ke beech price consistency checks
- Better evaluation with MAE, RMSE, R² and MAPE

**Abhi pehle:** Cell 1 se Cell 6 tak run karo. Agar 2.78 crore rows ka `df` load ho gaya, to agle step me isi notebook ko training-ready optimize karenge.